# Capítulo 4 — Armazenamento vetorial (Vector Storage)

**Livro:** começa na página **168** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## How are vectors stored · PDF p. 170

#### 🧪 Exercício: Um banco vetorial mínimo

Escreva a classe `BancoVetorial` com `adicionar(id, vetor, metadados)`, `buscar(vetor, k, filtro=None)` e `remover(id)`, guardando os vetores normalizados numa matriz NumPy. Teste com vetores TF-IDF dos documentos (sem precisar do modelo).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
class BancoVetorial:
    def __init__(self): self.ids, self.meta, self.M = [], [], None
    def adicionar(self, id_, v, meta):
        v = v / np.linalg.norm(v); self.ids.append(id_); self.meta.append(meta)
        self.M = v[None, :] if self.M is None else np.vstack([self.M, v])
    def remover(self, id_):
        i = self.ids.index(id_); self.ids.pop(i); self.meta.pop(i); self.M = np.delete(self.M, i, axis=0)
    def buscar(self, v, k=3, filtro=None):
        s = self.M @ (v / np.linalg.norm(v))
        ordem = [i for i in np.argsort(-s) if filtro is None or filtro(self.meta[i])]
        return [(self.ids[i], round(float(s[i]), 3)) for i in ordem[:k]]
vec = TfidfVectorizer().fit([d["texto"] for d in DOCS])
banco = BancoVetorial()
for d in DOCS: banco.adicionar(d["id"], vec.transform([d["texto"]]).toarray()[0], {"data": d["data"]})
q = vec.transform(["multa por atraso"]).toarray()[0]
print(banco.buscar(q)); print(banco.buscar(q, filtro=lambda m: m["data"] >= "2025")); banco.remover("d05"); print(banco.buscar(q))

## Indexing in vector databases · PDF p. 179

#### 🧪 Exercício: Índice IVF: buscar só nos grupos próximos

Gere 20.000 vetores aleatórios de dimensão 64. Agrupe com k-médias (100 grupos) e, para uma consulta, procure só nos 5 grupos mais próximos (ideia do IVF). Compare tempo e *recall@10* com a busca exata.

In [ ]:
import time
from sklearn.cluster import KMeans
rng = np.random.default_rng(0)
X = rng.normal(size=(20_000, 64)).astype("float32"); q = rng.normal(size=64).astype("float32")
km = KMeans(100, n_init=1, random_state=0).fit(X)
t = time.perf_counter(); exato = np.argsort(np.linalg.norm(X - q, axis=1))[:10]; t_exato = time.perf_counter() - t
t = time.perf_counter()
grupos = np.argsort(np.linalg.norm(km.cluster_centers_ - q, axis=1))[:5]
cand = np.where(np.isin(km.labels_, grupos))[0]
aprox = cand[np.argsort(np.linalg.norm(X[cand] - q, axis=1))[:10]]; t_ivf = time.perf_counter() - t
print(f"candidatos {len(cand)} | recall@10 {len(set(exato) & set(aprox)) / 10:.1f} | exato {t_exato*1000:.1f} ms | IVF {t_ivf*1000:.1f} ms")

## Vector retrieval (similarity measurement) · PDF p. 185

#### 🧪 Exercício: Métrica certa para o vetor certo

Mostre com um exemplo que, para vetores **não normalizados**, o ranking por produto escalar pode diferir do ranking por cosseno (um vetor longo "ganha" só por ser grande).

In [ ]:
q = np.array([1.0, 0.0])
docs = {"curto_alinhado": np.array([1.0, 0.1]), "longo_desalinhado": np.array([5.0, 4.0])}
for nome, v in docs.items():
    print(nome, "produto:", round(q @ v, 2), "cosseno:", round(q @ v / np.linalg.norm(v), 3))

## Data maintenance and vector storage CRUD operations in RAG systems · PDF p. 244

#### 🧪 Exercício: Atualizar documentos sem duplicar

Implemente um *upsert*: ao reindexar, se o `id` já existe e o texto mudou (compare um hash `hashlib.sha256`), substitua; se não mudou, ignore. Mostre o relatório de inseridos, atualizados e ignorados.

In [ ]:
import hashlib
indice = {}
def upsert(docs):
    rel = {"inseridos": 0, "atualizados": 0, "ignorados": 0}
    for d in docs:
        h = hashlib.sha256(d["texto"].encode()).hexdigest()
        if d["id"] not in indice: rel["inseridos"] += 1
        elif indice[d["id"]] != h: rel["atualizados"] += 1
        else: rel["ignorados"] += 1; continue
        indice[d["id"]] = h
    return rel
print(upsert(DOCS))
novos = [dict(d) for d in DOCS]; novos[2]["texto"] += " Agora também pelo aplicativo."
print(upsert(novos))